RAG Pipelines- Data Ingestion to vector DB pipelines

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\hp\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
### Read all the pdf's inside the directory
import os
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── 1. Load all PDFs from directory ──────────────────────────────────────────
pdf_dir = "../data/pdf"  # Change this to your actual path

dir_loader = DirectoryLoader(
    pdf_dir,
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

documents = dir_loader.load()
print(f"✅ Total pages loaded: {len(documents)}")
print(f"📄 Sample content:\n{documents[0].page_content[:300]}")

# ── 2. Split documents into chunks ───────────────────────────────────────────
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # Characters per chunk
    chunk_overlap=200,     # Overlap between chunks
    length_function=len
)

chunks = text_splitter.split_documents(documents)
print(f"\n✅ Total chunks created: {len(chunks)}")

# ── 3. Inspect a chunk ───────────────────────────────────────────────────────
print(f"\n📦 Sample chunk content:\n{chunks[0].page_content}")
print(f"\n🔖 Metadata: {chunks[0].metadata}")

##DirectoryLoader — recursively finds all .pdf files in the folder
##PyMuPDFLoader — fast and reliable PDF parser (better than PyPDFLoader)
##RecursiveCharacterTextSplitter — splits large pages into manageable chunks for embedding/retrieval
##metadata — each chunk automatically carries the source filename and page number


100%|██████████| 2/2 [00:00<00:00,  3.87it/s]

✅ Total pages loaded: 15
📄 Sample content:
SaaS
Q: What is SaaS in AWS?
A: SaaS (Software as a Service) is a cloud model where software applications are delivered over
the internet on a subscription basis.
Q: How does SaaS differ from PaaS and IaaS?
A: SaaS provides ready-to-use applications, PaaS provides a platform for developers to build 

✅ Total chunks created: 24

📦 Sample chunk content:
SaaS
Q: What is SaaS in AWS?
A: SaaS (Software as a Service) is a cloud model where software applications are delivered over
the internet on a subscription basis.
Q: How does SaaS differ from PaaS and IaaS?
A: SaaS provides ready-to-use applications, PaaS provides a platform for developers to build apps,
and IaaS provides infrastructure resources like servers and storage.
Q: Give an example of SaaS applications on AWS.
A: Examples include Amazon Chime, AWS WorkMail, and third-party SaaS solutions available on
AWS Marketplace.
Lambda
Q: What is AWS Lambda and why is it called serverless?
A: AWS La

In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Define the function first ─────────────────────────────────────────────────
def process_all_pdfs(data_dir: str, chunk_size: int = 1000, chunk_overlap: int = 200):
    """
    Recursively finds and loads all PDFs from the given directory.
    Returns a list of split document chunks.
    """
    data_path = Path(data_dir)
    
    # Find all PDF files recursively
    pdf_files = list(data_path.rglob("**/*.pdf"))
    
    if not pdf_files:
        print(f"⚠️ No PDF files found in: {data_path.resolve()}")
        return []
    
    print(f"📂 Found {len(pdf_files)} PDF(s):")
    for f in pdf_files:
        print(f"   - {f}")
    
    # Load each PDF
    all_documents = []
    for pdf_path in pdf_files:
        try:
            loader = PyMuPDFLoader(str(pdf_path))
            docs = loader.load()
            all_documents.extend(docs)
            print(f"✅ Loaded: {pdf_path.name} ({len(docs)} pages)")
        except Exception as e:
            print(f"❌ Failed to load {pdf_path.name}: {e}")
    
    print(f"\n📄 Total pages loaded: {len(all_documents)}")
    
    # Split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    chunks = text_splitter.split_documents(all_documents)
    print(f"📦 Total chunks created: {len(chunks)}")
    
    return chunks

# ── Now call it ───────────────────────────────────────────────────────────────
all_pdf_documents = process_all_pdfs("../data")
print(f"\n🔖 Sample metadata: {all_pdf_documents[0].metadata}")

📂 Found 2 PDF(s):
   - ..\data\pdf\AWS_Interview_QA.pdf
   - ..\data\pdf\NLP Lab Manual first four practicals.pdf
✅ Loaded: AWS_Interview_QA.pdf (3 pages)
✅ Loaded: NLP Lab Manual first four practicals.pdf (12 pages)

📄 Total pages loaded: 15
📦 Total chunks created: 24

🔖 Sample metadata: {'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-09-28T04:03:27+00:00', 'source': '..\\data\\pdf\\AWS_Interview_QA.pdf', 'file_path': '..\\data\\pdf\\AWS_Interview_QA.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-09-28T04:03:27+00:00', 'trapped': '', 'modDate': "D:20250928040327+00'00'", 'creationDate': "D:20250928040327+00'00'", 'page': 0}


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-09-28T04:03:27+00:00', 'source': '..\\data\\pdf\\AWS_Interview_QA.pdf', 'file_path': '..\\data\\pdf\\AWS_Interview_QA.pdf', 'total_pages': 3, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-09-28T04:03:27+00:00', 'trapped': '', 'modDate': "D:20250928040327+00'00'", 'creationDate': "D:20250928040327+00'00'", 'page': 0}, page_content='SaaS\nQ: What is SaaS in AWS?\nA: SaaS (Software as a Service) is a cloud model where software applications are delivered over\nthe internet on a subscription basis.\nQ: How does SaaS differ from PaaS and IaaS?\nA: SaaS provides ready-to-use applications, PaaS provides a platform for developers to build apps,\nand IaaS provides infrastructure resources like servers and storage.\nQ: Give an example of SaaS applications on AWS.\nA: Examples include Amaz

In [5]:
###Text splitting get  into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documets into smaller chunks for better RAG perfromance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split{len(documents)}documents into {len(split_docs)}chunks")
    #show example of chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

        return split_docs
    

    
    
              
               


In [8]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ✅ fixed
import os

# Step 1: Load PDFs
loader = DirectoryLoader(
    path="../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

all_pdf_documents = loader.load()
print(f"Loaded {len(all_pdf_documents)} pages")

# Step 2: Split
def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )
    return splitter.split_documents(documents)

chunks = split_documents(all_pdf_documents)
chunks

Loaded 15 pages


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-09-28T04:03:27+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-09-28T04:03:27+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\AWS_Interview_QA.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='SaaS\nQ: What is SaaS in AWS?\nA: SaaS (Software as a Service) is a cloud model where software applications are delivered over\nthe internet on a subscription basis.\nQ: How does SaaS differ from PaaS and IaaS?\nA: SaaS provides ready-to-use applications, PaaS provides a platform for developers to build apps,\nand IaaS provides infrastructure resources like servers and storage.\nQ: Give an example of SaaS applications on AWS.'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-09-28T04:03:27+00:00', '

In [6]:
import os
print(os.listdir("../data/pdf/"))  # Should show .pdf files

['AWS_Interview_QA.pdf', 'NLP Lab Manual first four practicals.pdf']


embedding and vectorStoreDB

In [7]:
import numpy as np
from sentence_transformers  import SentenceTransformer
import  chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict,  Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [8]:
class EmbeddingManager:
    """Handles documents embedding generatiion using SentenceTransformer"""
def _init_(self,model_name: str = "all-MiniLM-L6-v2"):
    """
    Intialize the embedding manager

    Args:
         model_name:  HuggingFace model  name for sentence embeddings
         """
    self.model_name = model_name
    self.model = None
    self._load_model()

def _load_model(self):
    """Load  the sentenceTransformer model"""
    try:
        print(f"Loading embedding model:{self.model_name}")
        self.model = SentenceTransformer(self.model_name)
        print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
    except Exception as e:
        print(f"Error loading model {self.model_name}: {e}")
        raise
    def generate_embeddings(self,texts: List[str])-> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: List of text strings to embed
        Returns:
            numpy array of embedings with shape (len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
    print(f"Generating embeddings for {len(texts)} texts...")
    embeddings = self.model.encode(texts, show_progress_bar=True)
    print(f"Generated embeddings with shape: {embeddings.shape}")
    return embeddings

## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager



    
    

        

VectorStore


In [9]:
class VectorStore:
    """Manage document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # ✅ Fixed: was missing parentheses — it's a method call, not assignment
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []   # ✅ Fixed: was 'embedding_list' (typo) — used 'embeddings_list' below

        # ✅ Fixed: this for loop was outside the class — now correctly indented inside add_documents
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        # ✅ Fixed: try/except was outside the loop — now correctly placed after loop
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


# ✅ Fixed: use a different variable name — 'VectorStore' was overwriting the class itself
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 40


In [21]:
#convert the text to embeddings
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

class EmbeddingManager:
    """Manage text embeddings using SentenceTransformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding model

        Args:
            model_name: SentenceTransformer model to use
        """
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.model_name = model_name
        print("Embedding model loaded successfully ✅")

    def generate_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed
            batch_size: Number of texts to process at once

        Returns:
            numpy array of embeddings
        """
        if not texts:
            raise ValueError("No texts provided for embedding")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        print(f"Embeddings generated ✅ — Shape: {embeddings.shape}")
        return embeddings

    def generate_single_embedding(self, text: str) -> np.ndarray:
        """Generate embedding for a single text (useful for query embedding)"""
        return self.model.encode([text], convert_to_numpy=True)[0]


In [22]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

# ── 1. Load Model ─────────────────────────────────────────────────────────────
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ Model loaded | Dimension: {model.get_sentence_embedding_dimension()}")

# ── 2. Extract texts from chunks ──────────────────────────────────────────────
texts = [doc.page_content for doc in chunks]
print(f"📄 Total texts to embed: {len(texts)}")

# ── 3. Generate Embeddings ────────────────────────────────────────────────────
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"✅ Embeddings generated!")
print(f"   Shape : {embeddings.shape}")        # (num_chunks, 384)
print(f"   Type  : {type(embeddings)}")
print(f"   Sample: {embeddings[0][:5]}")       # first 5 values of first embedding

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded | Dimension: 384
📄 Total texts to embed: 40


C:\Users\hp\AppData\Local\Temp\ipykernel_3388\2376675024.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"✅ Model loaded | Dimension: {model.get_sentence_embedding_dimension()}")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Embeddings generated!
   Shape : (40, 384)
   Type  : <class 'numpy.ndarray'>
   Sample: [-0.04558837 -0.02455154 -0.0479641  -0.01666448  0.01150703]


Retriever Pipeline From VectorStore



In [31]:
from typing import List, Dict, Any

class RAGRetriever:
    """Handle query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    # ✅ Fixed: retrieve() was indented inside __init__ — moved to class level
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        Args:
            query          : The search query
            top_k          : Number of top results to return
            score_threshold: Minimum similarity score threshold
        Returns:
            List of dicts containing retrieved documents and metadata
        """
        print(f"🔍 Query     : '{query}'")
        print(f"   Top K    : {top_k} | Score Threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            # ✅ Fixed: was 'results: self.vector_store...' (colon) → should be '=' (assignment)
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k       # ✅ Fixed: was top_K (wrong case)
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents  = results['documents'][0]
                metadatas  = results['metadatas'][0]
                distances  = results['distances'][0]
                ids        = results['ids'][0]

                # ✅ Fixed: 'enumesrate' typo → 'enumerate'
                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id'              : doc_id,
                            'content'         : document,   # ✅ Fixed: 'Content' → 'content' (consistent casing)
                            'metadata'        : metadata,
                            'similarity_score': similarity_score,
                            'distance'        : distance,
                            'rank'            : i + 1
                        })

            # ✅ Fixed: print & return were inside the for loop — moved outside
            if retrieved_docs:
                print(f"✅ Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("⚠️  No documents found above score threshold")

            return retrieved_docs

        except Exception as e:
            print(f"❌ Error during retrieval: {e}")
            return []


# ── Initialize & Test ─────────────────────────────────────────────────────────
rag_retriever = RAGRetriever(vector_store, embedding_manager)




In [32]:
rag_retriever

In [33]:
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

class EmbeddingManager:
    """Manages text embeddings using SentenceTransformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.model_name = model_name

        # ✅ Fix: use the updated method name
        dim = self.model.get_embedding_dimension()
        print(f"✅ Model loaded | Dimension: {dim}")

    def generate_embeddings(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not texts:
            raise ValueError("No texts provided")

        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True
        )
        return embeddings

    def generate_single_embedding(self, text: str) -> np.ndarray:
        """Generate embedding for a single query"""
        return self.model.encode([text], convert_to_numpy=True)[0]


# ── Reinitialize everything ───────────────────────────────────────────────────
embedding_manager = EmbeddingManager()
vector_store      = VectorStore()
rag_retriever     = RAGRetriever(vector_store, embedding_manager)


results = rag_retriever.retrieve("What is attention is all you need", top_k=3)

Loading embedding model: all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded | Dimension: 384
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 40
🔍 Query     : 'What is attention is all you need'
   Top K    : 3 | Score Threshold: 0.0
⚠️  No documents found above score threshold


In [34]:
rag_retriever.retrieve("Saas in AWS")

🔍 Query     : 'Saas in AWS'
   Top K    : 5 | Score Threshold: 0.0
✅ Retrieved 3 documents (after filtering)


[{'id': 'doc_7a5003a5_0',
  'content': 'SaaS\nQ: What is SaaS in AWS?\nA: SaaS (Software as a Service) is a cloud model where software applications are delivered over\nthe internet on a subscription basis.\nQ: How does SaaS differ from PaaS and IaaS?\nA: SaaS provides ready-to-use applications, PaaS provides a platform for developers to build apps,\nand IaaS provides infrastructure resources like servers and storage.\nQ: Give an example of SaaS applications on AWS.',
  'metadata': {'creator': '(unspecified)',
   'moddate': '2025-09-28T04:03:27+00:00',
   'page': 0,
   'producer': 'ReportLab PDF Library - www.reportlab.com',
   'keywords': '',
   'creationdate': '2025-09-28T04:03:27+00:00',
   'total_pages': 3,
   'source': '..\\data\\pdf\\AWS_Interview_QA.pdf',
   'author': '(anonymous)',
   'page_label': '1',
   'doc_index': 0,
   'content_length': 422,
   'trapped': '/False',
   'subject': '(unspecified)',
   'title': '(anonymous)'},
  'similarity_score': 0.540669322013855,
  'distan

Integration Vectordb Context pipeline with LLM output


In [37]:
def rag_simple(query, retriever, llm, top_k=3):
    # Step 1: Retrieve context using YOUR custom retriever
    results = retriever.retrieve(query, top_k=top_k)  # ✅ .retrieve(), not .invoke()

    # Step 2: Extract text from results
    # Adjust the key below based on what retriever.retrieve() actually returns
    if results and isinstance(results[0], dict):
        context = "\n\n".join([r.get("content", r.get("text", "")) for r in results])
    else:
        context = "\n\n".join([str(r) for r in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    # Step 3: Build prompt
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    # Step 4: Generate response
    response = llm.invoke(prompt)  # ✅ Pass string directly, no broken .format()
    return response.content


answer = rag_simple("What is attention mechanism?", rag_retriever, llm)
print(answer)

🔍 Query     : 'What is attention mechanism?'
   Top K    : 3 | Score Threshold: 0.0
⚠️  No documents found above score threshold
No relevant context found to answer the question.
